In [11]:
import os
from PIL import Image
from IPython.display import SVG, display
notebook_dir = os.path.abspath('')

In [27]:
def displayImage(img_path):
    try:
        with open(img_path, "r", encoding="utf-8") as f:
            svg_content = f.read()

        display(HTML(f"""
        <div style="background-color:#1e1e1e; padding:20px;">
            <div style="width:1000px;">
                {svg_content}
            </div>
        </div>
        """))

    except FileNotFoundError:
        print(f"Error: Could not find image at {img_path}")

# BERT Model Architecture: 
> Theory: https://youtu.be/90mGPxR2GgY?si=loAxdJ0z-a3Ov_nJ&t=1802 [Watch from: 30:02]

### Problem in BERT Model

**Problem 1** — [CLS] token is a poor sentence embedding[CLS] is like asking the "class monitor" to summarise a whole lecture, but they were only trained to take attendance. It wasn't trained on meaning — so the vector it produces is noisy and unreliable as a sentence representation.

In [28]:
img_path = os.path.join(notebook_dir, r"assets\bert_model\cls_poor_embedding.svg")
displayImage(img_path)

**Problem 2** — Cross-encoder cost explodes at scale
Think of it like this: to compare sentences with BERT, you can't look at them separately — you must put them in the same room and watch them interact. That's fine for 2 sentences, catastrophic for 10,000.

In [29]:
img_path = os.path.join(notebook_dir, r"assets\bert_model\cross_encoder_cost.svg")
displayImage(img_path)

**Problem 3** — No independent sentence vectors means no vector database
With BERT you can't ask "store this sentence's meaning and look it up later." Every comparison requires re-running the full model. There's no way to build an index.

In [30]:
img_path = os.path.join(notebook_dir, r"assets\bert_model\no_precompute_vectors.svg")
displayImage(img_path)

**Problem 4** — Anisotropy: all vectors point the same direction
Imagine trying to tell apart 10,000 arrows when 9,999 of them point almost identically north-east. The cosine similarity between any two random BERT sentence vectors is artificially high — making it useless for distinguishing similar from dissimilar sentences.

In [31]:
img_path = os.path.join(notebook_dir, r"assets\bert_model\anisotropy_embedding_space.svg")
displayImage(img_path)

**Problem 5** — No training signal for sentence similarity
BERT's two pre-training tasks never once asked "are these two sentences semantically close?" The model simply has no concept of sentence-level closeness baked in.

In [32]:
img_path = os.path.join(notebook_dir, r"assets\bert_model\bert_training_gap.svg")
displayImage(img_path)

# Sentence BERT (SBERT): A Comprehensive Guide

## Table of Contents
1. [Introduction to Sentence BERT](#introduction-to-sentence-bert)
2. [What is BERT and How SBERT Differs](#what-is-bert-and-how-sbert-differs)
3. [Key Components of Sentence BERT](#key-components-of-sentence-bert)
4. [Tokenization and Model Selection](#tokenization-and-model-selection)
5. [Training Process: Forward and Backward Passes](#training-process-forward-and-backward-passes)
6. [Loss Functions Explained](#loss-functions-explained)
7. [Pooling Strategies with Mathematics](#pooling-strategies-with-mathematics)
8. [Common Problems and Solutions](#common-problems-and-solutions)

---

## Introduction to Sentence BERT

Sentence BERT (SBERT) is an extension of the original BERT model specifically designed to produce meaningful sentence embeddings that can be efficiently compared using cosine similarity. While BERT is powerful for various NLP tasks, it wasn't originally optimized for generating sentence-level representations.

**Key Advantages of SBERT**:
- Generate sentence embeddings that capture semantic meaning
- Ideal for semantic similarity tasks
- Clustering and information retrieval
- Paraphrase detection
- Recommendation systems
- ~25x faster than BERT for similarity tasks


In [35]:
img_path = os.path.join(notebook_dir, r"assets\bert_model\sentence_bert_architecture.svg")
displayImage(img_path)


## What is BERT and How SBERT Differs

### Original BERT Model

BERT (Bidirectional Encoder Representations from Transformers) is a pre-trained transformer model that:
- Uses bidirectional context to understand language
- Produces token-level embeddings (one embedding per token)
- Requires paired input-output data for fine-tuning
- Doesn't naturally produce meaningful sentence-level embeddings

**BERT Output Challenge - An Analogy**:

Think of BERT like a sophisticated medical scanner that gives you detailed information about every cell in a patient's body. Each cell's analysis is highly precise and accurate. But when a doctor asks "What's the overall health status of the entire patient?", simply averaging all cell reports (or looking at just one cell) doesn't give a meaningful diagnosis about the whole organism.

Similarly, BERT gives you embeddings for each token in a sentence (like "[CLS]", "The", "movie", "was", "great", "[SEP]"), each 768-dimensional. When you naively average them or use just the [CLS] token:
- You lose important structural information about what makes sentences semantically similar
- The resulting "sentence embedding" isn't optimized for direct similarity comparison
- Cosine similarity between two sentence embeddings becomes unreliable and inconsistent

**Mathematical Problem with Naive Approach**:

For a sentence with tokens `t₁, t₂, ..., tₙ` with embeddings `e₁, e₂, ..., eₙ` (each 768-dimensional):

**Naive Averaging**: 
$$\text{sent\_emb} = \frac{1}{n}\sum_{i=1}^{n} e_i$$

**Problem**: This treats all tokens equally. Padding tokens [PAD], separator [SEP], punctuation contribute as much as semantic-rich tokens like nouns and verbs. There's no weighting for importance.

**Naive CLS Pooling**:
$$\text{sent\_emb} = e_{[CLS]}$$

**Problem**: Only the first token carries all sentence information. The [CLS] token in original BERT was trained for classification tasks, not to encode the entire semantic meaning of a sentence.

**The Core Issue - Similarity Unreliability**:

When you compute cosine similarity between two sentences using naive BERT embeddings:
$$\text{sim}(s_1, s_2) = \frac{e_{s_1} \cdot e_{s_2}}{||e_{s_1}|| \cdot ||e_{s_2}||}$$

This similarity score is:
- Not calibrated for semantic comparison
- Influenced by encoding artifacts rather than semantic content
- Unreliable when comparing sentences of different lengths

### Sentence BERT Solution

SBERT addresses these limitations through:

| Aspect | BERT | Sentence BERT |
|--------|------|---------------|
| **Architecture** | Transformer (unchanged) | Transformer + Pooling + Optional Projection |
| **Output Type** | Token-level (768-dim × n-tokens) | Sentence-level (384 or 768-dim) |
| **Training Goal** | MLM (Masked Language Modeling) + NSP | Minimize distance between similar sentences |
| **Loss Function** | Cross-entropy for predictions | Contrastive/Triplet/InfoNCE loss |
| **Inference Speed** | Slower for similarity comparison | ~25x faster for similarity tasks |
| **Similarity Metric** | Requires classification head | Direct cosine similarity |
| **Pooling Strategy** | Only [CLS] token | Mean/Max/CLS pooling + optional weighting |

**Key Insight**: SBERT is **not** a completely new architecture. It's BERT + Siamese Network Training + Optimized Pooling. The transformation comes from how you train it, not the underlying model.


---
## Key Components of Sentence BERT

### 1. **Siamese Network Architecture**

A Siamese network is a neural network architecture with a special property: **it contains two or more identical subnetworks that share weights**.

**Visual Representation**:
```
Input Pair: (Sentence A, Sentence B)
        ↓
    [Sentence A]            [Sentence B]
        ↓                        ↓
    BERT Model (shared)    BERT Model (shared)
        ↓                        ↓
    Token Embeddings      Token Embeddings
    (shape: 10×768)       (shape: 15×768)
        ↓                        ↓
    Pooling Layer         Pooling Layer
        ↓                        ↓
    Sentence Emb A        Sentence Emb B
    (384-dim)             (384-dim)
        ↓                        ↓
        └──────────────┬─────────┘
                       ↓
                Loss Computation
                (Triplet/Contrastive)
                       ↓
                Backpropagation
```

**Why Shared Weights Matter**:

The key principle is: **identical input pairs should produce identical embeddings**. With shared weights:
$$f_θ(\text{sentence}_A) = f_θ(\text{sentence}_B) \text{ if } \text{sentence}_A = \text{sentence}_B$$

Where `f_θ` is the BERT model with parameters `θ`.

Without shared weights, the two separate models could learn completely different representations for the same semantic content, defeating the purpose of comparison.

**The Metric Learning Perspective**:

In traditional machine learning, we optimize models to make good predictions. In Siamese networks, we optimize the **distance metric** between embeddings:

$$\text{Goal: Minimize} \quad d(f_θ(\text{similar\_pair}_1), f_θ(\text{similar\_pair}_2))$$
$$\text{Goal: Maximize} \quad d(f_θ(\text{dissimilar\_pair}_1), f_θ(\text{dissimilar\_pair}_2))$$

Where `d` is typically cosine distance or Euclidean distance.

### 2. **Pooling Strategies with Mathematical Details**

After BERT processes tokens, we convert token-level embeddings into a single sentence embedding. This is the **pooling** step.

**Notation**: Let's say BERT produces embeddings `E ∈ ℝ^(n×d)` where:
- `n` = number of tokens
- `d` = embedding dimension (768)
- `m` = attention mask (1 for real tokens, 0 for padding)

#### **Mean Pooling (Recommended)**
$$\text{sent\_emb} = \frac{1}{\sum_{i=1}^{n}m_i}\sum_{i=1}^{n}m_i \cdot E_i$$

**Explanation**: 
- Takes arithmetic mean of all real token embeddings (ignoring [PAD] tokens)
- `m_i` is the attention mask value (filters out padding)
- Dividing by `∑m_i` normalizes by the actual sentence length

**Practical Example**:
```
Sentence: "I love SBERT"
Tokens:   [CLS] I    love SBERT [SEP]
Embeddings: e₁  e₂   e₃   e₄    e₅  (each 768-dim)
Mask:      1   1    1    1      1

Mean Pool = (1×e₁ + 1×e₂ + 1×e₃ + 1×e₄ + 1×e₅) / 5
          = (e₁ + e₂ + e₃ + e₄ + e₅) / 5 ✓

Another sentence: "Good morning"
Tokens:   [CLS] Good morning [SEP] [PAD] [PAD] [PAD]
Embeddings: e₁  e₂  e₃       e₄   e₅   e₆    e₇
Mask:      1   1   1        1    0    0     0

Mean Pool = (1×e₁ + 1×e₂ + 1×e₃ + 1×e₄ + 0×e₅ + 0×e₆ + 0×e₇) / 4
          = (e₁ + e₂ + e₃ + e₄) / 4 ✓
          (padding tokens correctly excluded)
```

**Pros**: 
- ✅ Captures information from all tokens
- ✅ Balanced contribution from each token
- ✅ Mathematically well-behaved (gradient flow is smooth)

**Cons**: 
- ❌ Less emphasis on important tokens (all tokens weighted equally)
- ❌ Can be diluted by less important words

#### **Max Pooling**
$$\text{sent\_emb}_j = \max_{i=1}^{n}(E_{i,j}) \quad \text{for each dimension } j$$

**Explanation**: 
- For each dimension, take the maximum value across all tokens
- Results in one value per dimension

**Practical Example**:
```
Three tokens with embeddings (showing first 3 dimensions only):
Token 1 (e₁): [0.2, -0.5, 0.8, ...]
Token 2 (e₂): [0.1,  0.3, 0.2, ...]
Token 3 (e₃): [-0.1, 0.1, 0.5, ...]

Max Pool dimension 1: max(0.2, 0.1, -0.1) = 0.2
Max Pool dimension 2: max(-0.5, 0.3, 0.1) = 0.3
Max Pool dimension 3: max(0.8, 0.2, 0.5) = 0.8

Result: [0.2, 0.3, 0.8, ...]
```

**Pros**:
- ✅ Captures salient/extreme features
- ✅ Computationally simple

**Cons**: 
- ❌ Can lose information (most values ignored)
- ❌ Sensitive to outliers
- ❌ Gradient sparsity (gradients only flow through max values)
- ❌ Less stable training

#### **CLS Pooling**
$$\text{sent\_emb} = E_{[CLS]} = E_1$$

**Explanation**: Uses only the embedding of the [CLS] token (first token)

**Pros**:
- ✅ Computationally very efficient (O(1))
- ✅ Single forward pass needed
- ✅ Used by many pre-trained models

**Cons**: 
- ❌ Only one token carries all information
- ❌ May miss important semantic details
- ❌ Unless explicitly trained for it, [CLS] might not encode full sentence meaning

#### **Comparison and Selection**

| Pooling Type | Information Retention | Stability | Speed | Memory | Semantic Quality | Best For |
|--------------|----------------------|-----------|-------|--------|------------------|----------|
| Mean | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | General purpose, retrieval |
| Max | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | Feature extraction |
| CLS | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐ | Speed-critical systems |

**Recommendation**: **Mean pooling** is the standard choice for SBERT because it balances information retention with training stability.

In [ ]:
img_path = os.path.join(notebook_dir, r"assets\bert_model\pooling_strategies_comparison.svg")
displayImage(img_path)


---
## Tokenization and Model Selection

### Tokenizer Types for SBERT

Different SBERT models use different tokenizers. Understanding your tokenizer is crucial for reproducibility and performance.

#### **1. WordPiece Tokenization (BERT, SBERT base models)**

Used by: `sentence-transformers/all-MiniLM-L6-v2`, `sentence-transformers/all-mpnet-base-v2`

**Algorithm**: Iteratively finds longest subword in vocabulary

#### **2. SentencePiece Tokenization (XLM-RoBERTa, mBERT variants)**

Used by: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`

**Algorithm**: Learns subword units from data using Byte-Pair Encoding (BPE)

### Model Selection Guide

**For Different Use Cases**:

| Use Case | Recommended Model | Why |
|----------|-------------------|-----|
| General Semantic Similarity | `all-MiniLM-L6-v2` | Fast, small, high quality |
| Multilingual | `paraphrase-multilingual-mpnet-base-v2` | Supports 50+ languages |
| Scientific Papers | `allenai/specter` | Domain-specific training |
| Fast Inference | `all-MiniLM-L6-v2` | Only 22M parameters |
| Maximum Quality | `all-mpnet-base-v2` | 109M parameters, best accuracy |
| Long Documents | `sentence-transformers/multi-qa-mpnet-base-dot-product-v1` | Trained on long documents |


---
## Training Process: Forward and Backward Passes

### Complete Training Example: Triplet Loss

Let's walk through one complete training iteration with concrete numbers.

#### **Setup**

**Model**: SBERT based on `sentence-transformers/all-MiniLM-L6-v2`
- Embedding dimension: 384
- Uses mean pooling

**Batch Data**:
```
Anchor sentence:     "The cat is on the mat"
Positive sentence:   "The feline is on the rug"      (similar)
Negative sentence:   "Machine learning is fun"       (dissimilar)

Margin parameter:    m = 0.5
Learning rate:       lr = 2e-5
```

#### **Step 1: Forward Pass - Encode Anchor**

```
Input: "The cat is on the mat"

Tokenization:
[CLS] the cat is on the mat [SEP]
IDs: [101, 1996, 3368, 2003, 2006, 1996, 8318, 102]

BERT Processing:
Layer 1 → Layer 2 → ... → Layer 6 (MiniLM has 6 layers)

Token Embeddings Output (shape: 8 × 384):
Token 0 [CLS]:  [0.23, -0.15, 0.42, ..., 0.11]  (384 values)
Token 1 "the":  [0.18, 0.22, -0.33, ..., 0.09]
Token 2 "cat":  [0.51, -0.41, 0.28, ..., 0.44]
Token 3 "is":   [0.12, 0.33, 0.19, ..., 0.23]
Token 4 "on":   [0.40, -0.28, 0.31, ..., 0.15]
Token 5 "the":  [0.19, 0.21, -0.32, ..., 0.10]
Token 6 "mat":  [0.45, -0.35, 0.25, ..., 0.38]
Token 7 [SEP]:  [0.16, 0.18, -0.20, ..., 0.12]

Mean Pooling (all 8 tokens):
sent_emb_anchor[0] = (0.23 + 0.18 + 0.51 + 0.12 + 0.40 + 0.19 + 0.45 + 0.16) / 8
                   = 2.24 / 8 = 0.28
sent_emb_anchor[1] = (-0.15 + 0.22 + (-0.41) + 0.33 + (-0.28) + 0.21 + (-0.35) + 0.18) / 8
                   = -0.25 / 8 = -0.031
... (repeat for all 384 dimensions)

Final Anchor Embedding (384-dim):
sent_emb_anchor = [0.28, -0.031, ..., 0.19]  (384 dimensions)
```

#### **Step 2: Forward Pass - Encode Positive**

```
Input: "The feline is on the rug"

Tokenization:
[CLS] the feline is on the rug [SEP]
IDs: [101, 1996, 16769, 2003, 2006, 1996, 14916, 102]

BERT Processing (same weights as anchor):
... (similar process)

Token Embeddings Output (shape: 8 × 384):
Token 0 [CLS]:      [0.25, -0.12, 0.44, ..., 0.13]
Token 1 "the":      [0.20, 0.24, -0.31, ..., 0.08]
Token 2 "feline":   [0.49, -0.38, 0.30, ..., 0.42]
Token 3 "is":       [0.14, 0.35, 0.21, ..., 0.25]
Token 4 "on":       [0.42, -0.26, 0.33, ..., 0.17]
Token 5 "the":      [0.21, 0.23, -0.30, ..., 0.11]
Token 6 "rug":      [0.47, -0.33, 0.27, ..., 0.40]
Token 7 [SEP]:      [0.18, 0.20, -0.18, ..., 0.14]

Mean Pooling:
sent_emb_positive = [0.29, -0.019, ..., 0.21]  (384 dimensions)
```

#### **Step 3: Forward Pass - Encode Negative**

```
Input: "Machine learning is fun"

Tokenization:
[CLS] machine learning is fun [SEP]
IDs: [101, 2696, 4083, 2003, 4569, 102]

Token Embeddings Output (shape: 6 × 384):
Token 0 [CLS]:        [0.05, -0.42, 0.15, ..., 0.03]
Token 1 "machine":    [-0.12, 0.28, -0.25, ..., 0.08]
Token 2 "learning":   [0.31, -0.15, 0.44, ..., 0.22]
Token 3 "is":         [0.11, 0.30, 0.20, ..., 0.24]
Token 4 "fun":        [0.38, 0.15, 0.35, ..., 0.35]
Token 5 [SEP]:        [0.09, -0.08, 0.12, ..., 0.06]

Mean Pooling:
sent_emb_negative = [0.14, 0.018, ..., 0.16]  (384 dimensions)
```

#### **Step 4: Compute Distances**

Using **cosine distance**:
$$\text{cosine\_distance} = 1 - \text{cosine\_similarity}$$

$$\text{cosine\_similarity} = \frac{\vec{u} \cdot \vec{v}}{||\vec{u}|| \cdot ||\vec{v}||}$$

**Distance between Anchor and Positive**:
```
Dot product (first few dimensions):
(0.28 × 0.29) + (-0.031 × -0.019) + ... = 0.345 (full computation)

Norm of anchor: ||sent_emb_anchor|| = √(0.28² + (-0.031)² + ...) = 0.987
Norm of positive: ||sent_emb_positive|| = √(0.29² + (-0.019)² + ...) = 0.992

Cosine Similarity:
cos_sim(anchor, positive) = 0.345 / (0.987 × 0.992) = 0.352

Cosine Distance:
d(anchor, positive) = 1 - 0.352 = 0.648
```

**Distance between Anchor and Negative**:
```
Dot product: 0.156

Norm of negative: ||sent_emb_negative|| = 0.891

Cosine Similarity:
cos_sim(anchor, negative) = 0.156 / (0.987 × 0.891) = 0.178

Cosine Distance:
d(anchor, negative) = 1 - 0.178 = 0.822
```

#### **Step 5: Compute Loss (Triplet Loss)**

$$L = \max(0, \text{margin} + d(a, p) - d(a, n))$$

Where:
- `a` = anchor
- `p` = positive
- `n` = negative
- `margin` = 0.5

**Calculation**:
```
L = max(0, 0.5 + 0.648 - 0.822)
  = max(0, 0.326)
  = 0.326

This is a "moderate" loss. The model is learning:
- Anchor to positive distance: 0.648 (should be small) ✓ relatively good
- Anchor to negative distance: 0.822 (should be large) ✓ good
- Loss is positive but not huge (not 0, not very large)
```

**What this loss tells us**:
- The positive is closer to anchor than negative (good)
- But not by enough margin (0.5), so loss is positive
- The model will adjust weights to increase the distance gap further

#### **Step 6: Backward Pass - Computing Gradients**

We need to compute:
$$\frac{\partial L}{\partial \theta}$$

Where `θ` represents all model parameters.

**Chain of Computation**:
```
Loss L
  ↓ (derivative)
Distance d(a,n) and d(a,p)
  ↓ (chain rule)
Embeddings sent_emb_anchor, sent_emb_positive, sent_emb_negative
  ↓ (chain rule through pooling)
Token embeddings from BERT layers
  ↓ (chain rule through transformer layers)
Token embeddings from Layer 5
  ↓ (chain rule)
...
  ↓ (chain rule)
Token embeddings from Layer 1
  ↓ (chain rule through attention/feedforward)
Initial token embeddings (from input embeddings)
  ↓ (chain rule)
Model parameters θ
```

**Gradient of Loss with respect to Distances**:

$$\frac{\partial L}{\partial d(a,p)} = +1 \quad \text{(we want to increase this distance, so positive gradient)}$$

$$\frac{\partial L}{\partial d(a,n)} = -1 \quad \text{(we want to decrease this distance, so negative gradient)}$$

**Gradient of Distance with respect to Embeddings**:

For cosine distance: $\frac{\partial d(a,p)}{\partial \text{sent\_emb\_anchor}}$ is complex, involves:
- Dot product derivatives
- Norm derivatives
- Chain rule application

**Example (simplified)**:
```
If d(a,p) = 0.648 and we computed this from:
- Cosine similarity = 0.352
- Then cosine_distance = 1 - cos_sim

∂d/∂cos_sim = -1

∂cos_sim/∂dot_product = 1/(||a|| × ||p||) ≈ 1.02

∂cos_sim/∂||a|| = -(dot_product / (||a||²||p||)) ≈ -0.35

So ∂d/∂||a|| = ∂d/∂cos_sim × ∂cos_sim/∂||a|| 
              = (-1) × (-0.35) = 0.35

(The actual computation is very complex and involves backpropagation
through all embedding dimensions)
```

**Key Point**: Through backpropagation, we get gradients for every parameter in the BERT model, indicating how much each parameter contributed to the loss and in which direction to adjust.


---

## Loss Functions Explained

**Important Clarification**: Different loss functions are designed for different training scenarios. You typically use **ONE loss function** per training session, not all three simultaneously.

### 1. **Triplet Loss** (Most Common for SBERT)

**When to Use**: 
- When you have triplets of (anchor, positive, negative)
- Pairwise relationships are important
- You want to control margin explicitly

**Mathematical Definition**:
$$L_{\text{triplet}} = \max(0, m + d(a, p) - d(a, n))$$

Where:
- `a` = anchor sentence
- `p` = positive sentence (similar to anchor)
- `n` = negative sentence (dissimilar to anchor)
- `m` = margin (typically 0.5)
- `d` = distance metric (cosine or Euclidean)

**Interpretation**:
- Loss is zero if: `d(a,n) - d(a,p) ≥ m` (negative is sufficiently far from anchor)
- Loss increases if negative gets closer to anchor than margin allows
- Forces: negative_distance ≥ positive_distance + margin

**Example Scenario**:
```
Goal: "cat" should be similar to "feline" and dissimilar to "dog"

Anchor:   "cat sitting on chair"
Positive: "feline on furniture"     (semantically similar)
Negative: "dog running in park"     (semantically dissimilar)

Triplet Loss pulls positive closer to anchor and pushes negative away.
After training, similarity scores improve:
- sim(anchor, positive) → 0.8 (high similarity)
- sim(anchor, negative) → 0.2 (low similarity)
```

**Advantages**:
- ✅ Intuitive and well-understood
- ✅ Effective for retrieval tasks
- ✅ Explicit control via margin parameter
- ✅ Works well with contrastive learning

**Disadvantages**:
- ❌ Requires mining hard negatives (random negatives often too easy)
- ❌ Computationally expensive with large batches
- ❌ Sensitive to margin selection

---

### 2. **Contrastive Loss** (For Paired Comparisons)

**When to Use**:
- When you have paired data without explicit triplets
- Binary similarity labels (similar or dissimilar)
- One similarity score per pair

**Mathematical Definition**:
$$L_{\text{contrastive}} = y \cdot d(a, b)^2 + (1-y) \cdot \max(0, m - d(a, b))^2$$

Where:
- `a, b` = two embeddings
- `y` = label (1 if similar, 0 if dissimilar)
- `m` = margin (typically 1.0)
- `d` = distance metric

**Interpretation**:
- If y=1 (similar pair): Loss = d²(a,b) — penalizes large distances
- If y=0 (dissimilar pair): Loss = max(0, m - d(a,b))² — penalizes small distances below margin

**Example Scenario**:
```
Training pairs:
Pair 1: ("The weather is nice", "It's a beautiful day")         y=1 (similar)
        Distance after training: 0.2
        Loss_1 = 1 × 0.2² = 0.04 ✓ small loss

Pair 2: ("The cat slept", "Machine learning rocks")             y=0 (dissimilar)
        Distance after training: 0.6
        Loss_2 = (1-0) × max(0, 1 - 0.6)² = 0.4² = 0.16 ✓ acceptable

Pair 3: ("The cat slept", "The kitten napped")                  y=1 (similar)
        Distance after training: 0.8
        Loss_3 = 1 × 0.8² = 0.64 ✗ large loss (should be closer)

Pair 4: ("Hello world", "Programming is fun")                   y=0 (dissimilar)
        Distance after training: 0.3
        Loss_4 = (1-0) × max(0, 1 - 0.3)² = 0.7² = 0.49 ✗ large loss
                (should be farther, at least distance ≥ 1.0)
```

**Advantages**:
- ✅ Works with paired data (simpler than triplets)
- ✅ Explicit distinction between similar/dissimilar
- ✅ Easier to create training data

**Disadvantages**:
- ❌ Binary labels might oversimplify continuous similarity
- ❌ Margin has to be chosen correctly
- ❌ Less effective for retrieval than triplet loss

---

### 3. **Multiple Negative Ranking Loss (InfoNCE)** (Modern Standard)

**When to Use**:
- Large batch sizes available
- Training with in-batch negatives
- Most modern SBERT implementations

**Mathematical Definition**:

For a batch with pairs (positive anchor-positive pairs):
$$L = -\log \frac{e^{s(a,p)/\tau}}{\sum_{i=0}^{n}e^{s(a,n_i)/\tau}}$$

Where:
- `a` = anchor
- `p` = positive sample
- `n_i` = all other samples in batch (serve as negatives)
- `s` = cosine similarity
- `τ` = temperature (typically 0.05)
- `n` = batch size

**Interpretation**:
- Numerator: similarity score between anchor and its positive
- Denominator: sum of similarities between anchor and all samples (positive + in-batch negatives)
- Loss is minimized when positive has highest similarity in the batch

**Example Scenario**:
```
Batch size = 4
Pairs: (anchor_1, positive_1), (anchor_2, positive_2), (anchor_3, positive_3)

After encoding, compute similarity matrix (4×4):
                pos_1    pos_2    pos_3    anchor_4
anchor_1        0.78     0.15     0.22     0.18
anchor_2        0.12     0.82     0.19     0.21
anchor_3        0.16     0.18     0.85     0.20
anchor_4        0.14     0.22     0.19     0.75  (diagonal for pairing)

For anchor_1:
- Similarity with its positive (pos_1): 0.78 ✓ high
- Similarities with other samples: 0.15, 0.22, 0.18 (in-batch negatives)

Loss for anchor_1:
L = -log(exp(0.78/0.05) / (exp(0.78/0.05) + exp(0.15/0.05) + exp(0.22/0.05) + exp(0.18/0.05)))
  = -log(exp(15.6) / (exp(15.6) + exp(3) + exp(4.4) + exp(3.6)))
  = -log(exp(15.6) / (exp(15.6) + 20.09 + 81.45 + 36.60))
  ≈ -log(10854.7 / 10992.8)
  ≈ -log(0.987)
  ≈ 0.013 (very small, good!)

(If positive_1 had similarity 0.5 instead of 0.78:
L = -log(exp(10) / (exp(10) + 20.09 + 81.45 + 36.60))
  ≈ -log(22026.5 / 22164.6)
  ≈ 0.006  (still acceptable but larger)
)
```

**Temperature Parameter Explanation**:
```
τ = 0.05 (small): Sharpens the probability distribution
    exp(0.78/0.05) = exp(15.6) = very large
    exp(0.15/0.05) = exp(3) = much smaller
    → Positive must have much higher similarity than negatives

τ = 1.0 (large): Softens the distribution
    exp(0.78/1.0) = exp(0.78) = 2.18
    exp(0.15/1.0) = exp(0.15) = 1.16
    → Less strict separation needed
    
τ = 0.01 (very small): Extremely sharp
    Requires very high sim(positive) relative to negatives
    More aggressive training, might overfit
```

**Advantages**:
- ✅ Efficiently uses entire batch (every sample as potential negative)
- ✅ Scales well to large batches
- ✅ No hard negative mining needed
- ✅ Modern and theoretically sound (information theory grounded)
- ✅ Used by latest SBERT implementations

**Disadvantages**:
- ❌ Requires specific pairing structure
- ❌ Less effective with small batches
- ❌ Sensitive to temperature parameter

**The Original SBERT Paper (2019)**:
- Used **Triplet Loss** initially
- Later papers showed **Contrastive Loss** works well too
- Current implementations prefer **Multiple Negative Ranking Loss** (InfoNCE variant)

---

## Summary Table: Complete Comparison

| Aspect | Triplet Loss | Contrastive Loss | Multi Neg Ranking |
|--------|--------------|------------------|-------------------|
| **Data Format** | Triplets (a,p,n) | Pairs with labels | Pairs in batches |
| **Batch Dependency** | No (independent) | No | Yes (critical) |
| **Efficiency** | Medium | Low | High |
| **Stability** | High | Medium | High |
| **Implementation** | Simple | Simple | Medium |
| **Performance** | Good | Good | Excellent |
| **Recommended Batch Size** | 16-64 | Any | 64+ |
| **Hard Negative Mining** | Yes (needed) | No | No (automatic) |
| **Current Popularity** | Declining | Stable | Rising |


In [37]:
img_path = os.path.join(notebook_dir, r"assets\bert_model\loss_functions_visualization.svg")
displayImage(img_path)
